#Phase 1: Environment Setup & Data Pipeline



In [9]:
# ── Cell 1: Install ───────────────────────────────────────────────
!pip install -q torch transformers datasets accelerate peft trl bitsandbytes
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
# ── Cell 2: hardware + CUDA smoke test + versions ─────────
import torch, transformers, peft, trl, datasets, bitsandbytes as bnb

print("=== HARDWARE ===")
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU:      {gpu.name}")
print(f"VRAM:     {gpu.total_memory / 1024**3:.1f} GB")
print(f"CUDA:     {torch.version.cuda}")
print(f"bf16 OK?: {torch.cuda.is_bf16_supported()}")

# Smoke test: force an actual GPU computation. If torch/driver mismatch,
# THIS is where it errors — better here than mid-training.
x = (torch.randn(1000, device="cuda") @ torch.randn(1000, device="cuda")).item()
print("CUDA compute OK ")

print("\n=== VERSIONS (requirements.txt) ===")
for lib in (torch, transformers, peft, trl, datasets, bnb):
    print(f"{lib.__name__:14s} {lib.__version__}")

=== HARDWARE ===
GPU:      Tesla T4
VRAM:     14.6 GB
CUDA:     12.8
bf16 OK?: True
CUDA compute OK 

=== VERSIONS (requirements.txt) ===
torch          2.11.0+cu128
transformers   5.13.1
peft           0.19.1
trl            1.9.2
datasets       5.0.1
bitsandbytes   0.50.0


In [3]:
# ── Cell 3: Load + inspect ────
from datasets import load_dataset

raw = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")

print(raw)          # shows features (column names) + row count
print("---")
print(raw[0])       # shows one real example so we see the actual field names

Dataset({
    features: ['qtype', 'Question', 'Answer'],
    num_rows: 16407
})
---
{'qtype': 'susceptibility', 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?', 'Answer': 'LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'}


In [4]:
# ── Cell 3b: measure answer length so MAX_LEN is a decision, not a guess ──
from transformers import AutoTokenizer
tok_probe = AutoTokenizer.from_pretrained("gpt2")

# token length of prompt+answer for a sample of rows
import numpy as np
lens = [len(tok_probe(f"### Instruction:\n{r['Question']}\n\n### Response:\n{r['Answer']}")["input_ids"])
        for r in raw.select(range(2000))]
lens = np.array(lens)
for p in (50, 90, 95, 99):
    print(f"{p}th percentile: {np.percentile(lens, p):.0f} tokens")
print(f"max: {lens.max()}  |  % over 512: {(lens > 512).mean()*100:.1f}%")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2963 > 1024). Running this sequence through the model will result in indexing errors


50th percentile: 171 tokens
90th percentile: 597 tokens
95th percentile: 987 tokens
99th percentile: 1995 tokens
max: 5043  |  % over 512: 12.3%


In [5]:
# ── Cell 4: Format with instruction template + reproducible split ──
PROMPT_TEMPLATE = "### Instruction:\n{q}\n\n### Response:\n"

def format_example(ex):
    prompt = PROMPT_TEMPLATE.format(q=ex["Question"].strip())
    return {"prompt": prompt, "text": prompt + ex["Answer"].strip()}

# drop qtype/Question/Answer, keep only prompt + text
dataset = raw.map(format_example, remove_columns=raw.column_names)

splits   = dataset.train_test_split(test_size=0.1, seed=42)   # seed => reproducible eval set
train_ds, eval_ds = splits["train"], splits["test"]
print(f"train={len(train_ds)}  eval={len(eval_ds)}")
print(train_ds[0]["text"][:400])

train=14766  eval=1641
### Instruction:
What are the treatments for Lennox-Gastaut syndrome ?

### Response:
These resources address the diagnosis or management of Lennox-Gastaut syndrome:  - Cleveland Clinic  - Genetic Testing Registry: Epileptic encephalopathy Lennox-Gastaut type  - National Institute of Neurological Disorders and Stroke: Diagnosis and Treatment of Epilepsy  - News Release: FDA Approves New Drug to Tr


Where to set MAX_LEN, given that ceiling

So our realistic options for GPT-2 are bounded at 1024. Weigh them:

512 → covers ~88% of examples fully; truncates 12.3%. VRAM-cheap, fast.
768 → covers ~93%; truncates ~7%. Middle ground.
1024 (GPT-2's max) → covers ~95%+; truncates only the ~5% above 987 tokens. Costs the most memory/compute per step.

Here's my recommendation, and the reasoning: go with 1024. Two justifications. First, it's the most GPT-2 can do, so it gives your baseline its fair best shot — you don't want to handicap the baseline and then "win" with LLaMA unfairly; a clean experiment gives both models their honest best. Second, we control the memory cost not by shrinking MAX_LEN but with batch size + gradient accumulation (small per_device_train_batch_size, larger gradient_accumulation_steps) — that's the proper lever, and it's exactly the hardware-management skill Phase 2 is about. We keep the effective batch size healthy while the physical batch stays tiny enough for 14.6 GB.

The remaining ~5% of truly enormous answers (>1024) will get truncated regardless — that's an unavoidable consequence of GPT-2's architecture, and we simply document it as a known limitation.

Decision locked: MAX_LEN = 1024 for GPT-2.

In [6]:
# ── Cell 5 : Tokenizer + don't pre-build labels — let the collator do it ──
from transformers import AutoTokenizer

MODEL_NAME, MAX_LEN = "gpt2", 1024
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_fn(ex):
    out = tokenizer(
        ex["text"] + tokenizer.eos_token,   # EOS -> teaches the model to STOP
        truncation=True, max_length=MAX_LEN,
        padding=False,                       # collator pads per-batch
    )
    return out                               # <-- NO manual labels; collator builds them

train_tok = train_ds.map(tokenize_fn, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(tokenize_fn,  remove_columns=eval_ds.column_names)
print("example token length:", len(train_tok[0]["input_ids"]))
print("keys:", list(train_tok[0].keys()))   # expect input_ids, attention_mask (NO labels)

example token length: 163
keys: ['input_ids', 'attention_mask']


In [7]:
# ── Cell 6: Dynamic-padding collator + Phase 1 sanity check ───────
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)  # causal, not masked
batch = next(iter(DataLoader(train_tok, batch_size=4, collate_fn=collator)))

print("input_ids   :", batch["input_ids"].shape)      # [4, longest-in-batch]
print("attn_mask   :", batch["attention_mask"].shape)
print("pad in row0 :", (batch["attention_mask"][0] == 0).sum().item())
print("labels==-100:", (batch["labels"] == -100).sum().item())  # should EQUAL total pad count

input_ids   : torch.Size([4, 551])
attn_mask   : torch.Size([4, 551])
pad in row0 : 388
labels==-100: 827


#Phase 2: Baseline Fine-Tuning with GPT-2 (LoRA)

In [ ]:
# ── Cell 7: Load GPT-2, attach LoRA, verify the attach ────────────


In [8]:
# ── Cell 7: Load GPT-2 + attach LoRA + verify ──
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=["c_attn", "c_proj"],   # GPT-2 projections
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 1,622,016 || all params: 126,061,824 || trainable%: 1.2867


In [9]:
# ── Cell 8: Train GPT-2 + LoRA (T4-aware) ─────────────────────────
import torch
from transformers import Trainer, TrainingArguments

use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8   # False on T4 -> fp16
model.config.use_cache = False   # required once we train; re-enable for generation later

args = TrainingArguments(
    output_dir="gpt2-medqa-lora",
    num_train_epochs=1,
    per_device_train_batch_size=8,      # physical batch — fits the T4
    gradient_accumulation_steps=2,      # -> effective batch = 16
    learning_rate=2e-4,                 # LoRA tolerates a higher LR
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=25,                   # watch the loss curve
    save_strategy="epoch",
    report_to="none",                   # no wandb prompt
    fp16=not use_bf16,                  # T4 -> True
    bf16=use_bf16,                      # Ampere+ -> True
    optim="adamw_torch",                # 8-bit paging saved for Phase 3
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collator,             # dynamic padding + label masking (verified Phase 1)
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
25,2.832053
50,2.594648
75,2.418130
100,2.317937
125,2.230004
150,2.092147
175,2.103587
200,2.069000
225,2.020617
250,2.028007


Step,Training Loss
25,2.832053
50,2.594648
75,2.418130
100,2.317937
125,2.230004
150,2.092147
175,2.103587
200,2.069000
225,2.020617
250,2.028007


TrainOutput(global_step=923, training_loss=2.005529447082321, metrics={'train_runtime': 1180.277, 'train_samples_per_second': 12.511, 'train_steps_per_second': 0.782, 'total_flos': 5306331753566208.0, 'train_loss': 2.005529447082321, 'epoch': 1.0})

In [10]:
# ── Cell 9: Held-out evaluation -> baseline perplexity ────────────
import math

eval_metrics = trainer.evaluate()          # forward pass over eval_tok (unseen data)
eval_loss = eval_metrics["eval_loss"]
ppl = math.exp(eval_loss)

print(f"GPT-2 baseline eval loss : {eval_loss:.4f}")
print(f"GPT-2 baseline perplexity: {ppl:.2f}")

Training Loss,Validation Loss,Step
1.892809,1.789961,923


GPT-2 baseline eval loss : 1.7900
GPT-2 baseline perplexity: 5.99


In [11]:
# ── Cell 10: Persist the adapter (tiny — just the B·A matrices) ───
trainer.save_model("gpt2-medqa-lora")       # saves LoRA adapter + config
tokenizer.save_pretrained("gpt2-medqa-lora")
!du -sh gpt2-medqa-lora                      # note how small a LoRA adapter is (~6 MB)

32M	gpt2-medqa-lora


#Phase 3: Parameter-Efficient Fine-Tuning with LLaMA (LoRA + QLoRA)

In [ ]:
#Load the model in 4-bit + attach QLoRA, and prove it fits

In [12]:
# ── Cell 11: TinyLlama tokenizer + native chat template ──
# (If you restarted since Phase 1, re-run Cells 3–4 first so train_ds/eval_ds exist.)
from transformers import AutoTokenizer
MODEL_ID, MAX_LEN = "TinyLlama/TinyLlama-1.1B-Chat-v1.0", 1024
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

assert tokenizer.chat_template is not None, "No template shipped — tell me"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(tokenizer.chat_template[:300])

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

{% for message in messages %}
{% if message['role'] == 'user' %}
{{ '<|user|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'system' %}
{{ '<|system|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'assistant' %}
{{ '<|assistant|>
'  + message['content'] + eos_to


In [ ]:
# ── Cell 12: prove it fits ────────────────────────────────────────
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [13]:
# ── Cell 12: format the SAME data with TinyLlama's chat template ──
def to_chat_text(ex):
    q = ex["prompt"].replace("### Instruction:\n", "").replace("\n\n### Response:\n", "")
    a = ex["text"].split("### Response:\n", 1)[-1]
    msgs = [{"role": "user", "content": q.strip()},
            {"role": "assistant", "content": a.strip()}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False,
                                                  add_generation_prompt=False)}

train_chat = train_ds.map(to_chat_text, remove_columns=train_ds.column_names)
eval_chat  = eval_ds.map(to_chat_text,  remove_columns=eval_ds.column_names)
print(train_chat[0]["text"][:300])   # should open with <|user|>

Map:   0%|          | 0/14766 [00:00<?, ? examples/s]

Map:   0%|          | 0/1641 [00:00<?, ? examples/s]

<|user|>
What are the treatments for Lennox-Gastaut syndrome ?</s>
<|assistant|>
These resources address the diagnosis or management of Lennox-Gastaut syndrome:  - Cleveland Clinic  - Genetic Testing Registry: Epileptic encephalopathy Lennox-Gastaut type  - National Institute of Neurological Disorde


In [14]:
# ── Cell 13: 4-bit QLoRA load + LoRA attach (fp16 compute, fp32 adapters) ──
import torch, gc
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

try:
    del model, trainer          # free the GPT-2 model from Phase 2
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

COMPUTE_DTYPE = torch.float16   # T4 = Turing (7.5): NO bf16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", torch_dtype=COMPUTE_DTYPE)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=["q_proj","k_proj","v_proj","o_proj"])
model = get_peft_model(model, lora_config)
model.config.pad_token_id = tokenizer.pad_token_id
# adapters stay fp32 (default); Cell 15 trains with NO scaler -> no dtype collision
model.print_trainable_parameters()   # expect ~0.41% / 4,505,600

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [15]:
# ── Cell 14: prove it fits ──
import torch
print("VRAM allocated:", f"{torch.cuda.memory_allocated()/1024**3:.2f} GB")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

VRAM allocated: 1.01 GB
memory.used [MiB], memory.total [MiB]
1457 MiB, 15360 MiB


In [16]:
# ── Cell 15: Train TinyLlama + QLoRA (fp32 adapters, no scaler) ──
from trl import SFTConfig, SFTTrainer
sft_config = SFTConfig(
    output_dir="tinyllama-medqa-qlora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,       # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=30,
    logging_steps=25,
    save_strategy="epoch",
    report_to="none",
    fp16=False, bf16=False,              # no scaler -> no bf16/fp16 gradient crash
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_length=1024,
    dataset_text_field="text",
)
trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_chat, eval_dataset=eval_chat,
    processing_class=tokenizer)
trainer.train()

Adding EOS to train dataset:   0%|          | 0/14766 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/14766 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2206 > 2048). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/14766 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14766 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/1641 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
25,1.683115
50,1.352300


Step,Training Loss
25,1.683115
50,1.352300
75,1.169135
100,1.152282
125,1.075987
150,1.059896
175,1.105977
200,1.099202
225,1.072346
250,1.088328


TrainOutput(global_step=923, training_loss=1.0693219454549374, metrics={'train_runtime': 9255.4663, 'train_samples_per_second': 1.595, 'train_steps_per_second': 0.1, 'total_flos': 5.45961350854656e+16, 'train_loss': 1.0693219454549374, 'entropy': 1.0054241221236146, 'num_tokens': 4845928.0, 'mean_token_accuracy': 0.7648182867661767, 'epoch': 1.0})

In [17]:
# ── Cell 16: TinyLlama held-out perplexity vs GPT-2 ──
import math
metrics = trainer.evaluate()
print(f"TinyLlama eval loss : {metrics['eval_loss']:.4f}")
print(f"TinyLlama perplexity: {math.exp(metrics['eval_loss']):.2f}")
print("GPT-2 baseline was  : 5.99")

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
1.046381,1.029341,923,1.054099,4845928.000000,0.751374


TinyLlama eval loss : 1.0293
TinyLlama perplexity: 2.80
GPT-2 baseline was  : 5.99


In [18]:
# ── Cell 17: persist the QLoRA adapter ──
trainer.save_model("tinyllama-medqa-qlora")
tokenizer.save_pretrained("tinyllama-medqa-qlora")
!du -sh tinyllama-medqa-qlora

34M	tinyllama-medqa-qlora


In [19]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r tinyllama-medqa-qlora "/content/drive/MyDrive/tinyllama-medqa-qlora"
!cp -r gpt2-medqa-lora "/content/drive/MyDrive/gpt2-medqa-lora"
print("✅ both adapters backed up to Drive")

Mounted at /content/drive
✅ both adapters backed up to Drive


In [20]:
# ── Cell 18: free training state before loading both models for inference ──
import torch, gc
try:
    del model, trainer
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()
print("VRAM after cleanup:", f"{torch.cuda.memory_allocated()/1024**3:.2f} GB")

VRAM after cleanup: 0.26 GB


In [21]:
# ── Cell 19: modular inference class (Phase 5 module) ──
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

class DomainChatModel:
    """Loads a base model + saved LoRA adapter for generation.
       kind='gpt2' -> ### Instruction template; kind='chat' -> chat template."""

    def __init__(self, base_id, adapter_dir, kind, load_in_4bit=False):
        self.kind = kind
        self.tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"          # generation default (see note above)

        if load_in_4bit:                               # TinyLlama: keep it in 4-bit
            bnb = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
            base = AutoModelForCausalLM.from_pretrained(
                base_id, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
        else:                                          # GPT-2: small, plain fp16
            base = AutoModelForCausalLM.from_pretrained(
                base_id, torch_dtype=torch.float16, device_map="auto")

        self.model = PeftModel.from_pretrained(base, adapter_dir)  # snap adapter on
        self.model.eval()
        self.model.config.use_cache = True             # re-enable KV cache for fast gen
        self.device = next(self.model.parameters()).device

    def build_prompt(self, question):
        if self.kind == "gpt2":
            return f"### Instruction:\n{question.strip()}\n\n### Response:\n"
        msgs = [{"role": "user", "content": question.strip()}]
        return self.tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)   # appends <|assistant|>

    @torch.no_grad()
    def generate(self, question, max_new_tokens=256):
        prompt = self.build_prompt(question)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        out = self.model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,   # sampled, mild diversity
            repetition_penalty=1.15,                      # small models loop without this
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        new = out[0][inputs["input_ids"].shape[1]:]       # slice off the prompt echo
        return self.tokenizer.decode(new, skip_special_tokens=True).strip()

In [22]:
# ── Cell 20: load both fine-tuned models ──
gpt2_chat = DomainChatModel(
    "gpt2", "gpt2-medqa-lora", kind="gpt2", load_in_4bit=False)
tinyllama_chat = DomainChatModel(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "tinyllama-medqa-qlora",
    kind="chat", load_in_4bit=True)
print("both models loaded ✅  VRAM:", f"{torch.cuda.memory_allocated()/1024**3:.2f} GB")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

both models loaded ✅  VRAM: 1.24 GB


In [23]:
# ── Cell 21: qualitative side-by-side on HELD-OUT questions ──
# needs eval_ds from Phase 1 (re-run Cells 3–4 if you restarted).
import textwrap
def show(s, n=380): return textwrap.shorten(s.replace("\n", " "), n)

for ex in eval_ds.select(range(3)):
    q   = ex["prompt"].replace("### Instruction:\n","").replace("\n\n### Response:\n","").strip()
    ref = ex["text"].split("### Response:\n", 1)[-1].strip()
    print("="*90)
    print("Q:", q)
    print("-"*90)
    print("GPT-2     :", show(gpt2_chat.generate(q)))
    print("TinyLlama :", show(tinyllama_chat.generate(q)))
    print("Reference :", show(ref))

Q: Is D-bifunctional protein deficiency inherited ?
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GPT-2     : Determining the cause of this disorder is difficult. The genetic abnormalities are unknown, and no one has proven that d-biloquine affects other proteins in the body. However a recent study by University College London scientists found that mutations associated with an enzyme called CRISPR (CRF1) had been discovered on chromosome 9a4 genes related to ADH2 production at [...]
TinyLlama : This condition is inherited in an autosomal recessive pattern, which means both copies of the gene in each cell have mutations. The parents of an individual with an autosomal recessive condition each carry one copy of the mutated gene, but they typically do not show signs and symptoms of the condition.
Reference : This condition is inherited in an autosomal recessive pattern, which means both copies of the gene in each cell have mutations. The parents of an individual with an autosomal recessive condition each carry one copy of the mutated gene, but they typically do not show signs and sympt

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GPT-2     : Tourettes syndrome is a rare neurological disorder that affects the frontal lobes, or lower back. It's usually caused by an abnormal mutation in another gene called ALA1/ALB2. In people with this condition it can cause tremors and other symptoms such as muscle weakness, numbness of hands, chest pain, dizziness; seizures; difficulty speaking but no memory for words; speech [...]
TinyLlama : Tourette syndrome is a disorder that affects the nervous system. It causes tics and vocalizations in some people, although not all affected individuals have these symptoms. People with tourette syndrome can also experience other problems such as eye movements, difficulty walking, and problems with speech. Symptoms usually begin during childhood or adolescence and may [...]
Reference : Tourette syndrome is a complex disorder characterized by repetitive, sudden, and involuntary movements or noises called tics. Tics usually appear in childhood, and their severity varies over time. In most ca

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GPT-2     : Compulsory gambling, a type of behavior that involves betting and using illegal drugs or alcohol to win money from others, can be considered addiction. People who gamble on behalf other people are not addicts but risk losing their lives by doing so without knowing the consequences. The term "compulsive gambler" comes from the Greek word for addicted, meaning "to cheat." [...]
TinyLlama : Compulsive gambling involves a person's inability to resist the urge to gamble. It can lead to financial problems, addiction, and legal trouble. Many people who have compulsive gambling also suffer from depression or other mental health disorders. The most common type of compulsive gambling is pathological gambling, which occurs when a person loses control over their [...]
Reference : Many people enjoy gambling, whether it's betting on a horse or playing poker on the Internet. Most people who gamble don't have a problem, but some lose control of their gambling. Signs of problem gambling inc

In [24]:
# ── Cell 22: quantitative summary + % improvement ──
gpt2_ppl, tiny_ppl = 5.99, 2.80          # measured in Cells 9 and 16
reduction = (gpt2_ppl - tiny_ppl) / gpt2_ppl * 100
print(f"{'Model':<12}{'Perplexity':>12}")
print(f"{'GPT-2':<12}{gpt2_ppl:>12.2f}")
print(f"{'TinyLlama':<12}{tiny_ppl:>12.2f}")
print(f"\nTinyLlama reduces held-out perplexity by {reduction:.1f}% vs the GPT-2 baseline.")

Model         Perplexity
GPT-2               5.99
TinyLlama           2.80

TinyLlama reduces held-out perplexity by 53.3% vs the GPT-2 baseline.


In [25]:
# ── Cell 23: authenticate to the Hub with a WRITE token ──
# Create one at huggingface.co/settings/tokens (role: Write), then paste when prompted.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# ── Cell 24: push both adapters as their own model repos ──
from huggingface_hub import whoami
user = whoami()["name"]
print("pushing as:", user)

# push_to_hub creates the repo and uploads adapter weights + config + tokenizer.
gpt2_chat.model.push_to_hub(f"{user}/gpt2-medqa-lora")
gpt2_chat.tokenizer.push_to_hub(f"{user}/gpt2-medqa-lora")

tinyllama_chat.model.push_to_hub(f"{user}/tinyllama-medqa-qlora")
tinyllama_chat.tokenizer.push_to_hub(f"{user}/tinyllama-medqa-qlora")

print(f"✅ pushed:\n  {user}/gpt2-medqa-lora\n  {user}/tinyllama-medqa-qlora")

pushing as: Babblu2821


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   9%|8         |  555kB / 6.50MB            